## Project -> Question & Answering on Private Documents

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

### 1. Load Documents from a file and online

In [2]:
def load_documents(file):
    name, extension = os.path.splitext(file)
    if extension == ".pdf":
        print(f"Loading PDF document: {name}")
        from langchain_community.document_loaders import PyPDFLoader
        loader = PyPDFLoader(file)
    elif extension == ".docx":
        print(f"Loading Word document: {name}")
        from langchain_community.document_loaders import Docx2txtLoader
        loader = Docx2txtLoader(file)
    else:
        print("Unable to load the file")
        return None
    data = loader.load()
    return data

In [3]:
from langchain_community.document_loaders import WikipediaLoader
def load_from_wikipedia(query, max_docs=2, lang='en'):
    print(f"Loading document from WIkipedia for {query}")
    loader = WikipediaLoader(query=query, 
                           load_max_docs=max_docs,
                           lang=lang
                          )
    data = loader.load()
    return data

In [4]:
data = load_documents('files/us_constitution.pdf')

Loading PDF document: files/us_constitution


In [5]:
print(data[1].page_content)

TheHouseofRepresentativesshallbecomposedofMemberschosen
everysecondYearbythePeopleoftheseveralStates,andthe
ElectorsineachStateshallhavetheQualificationsrequisitefor
ElectorsofthemostnumerousBranchoftheStateLegislature.
NoPersonshallbeaRepresentativewhoshallnothaveattainedtothe
AgeoftwentyfiveYears,andbeensevenYearsaCitizenoftheUnited
States,andwhoshallnot,whenelected,beanInhabitantofthatState
inwhichheshallbechosen.
RepresentativesanddirectTaxesshallbeapportionedamongthe
severalStateswhichmaybeincludedwithinthisUnion,accordingto
theirrespectiveNumbers,whichshallbedeterminedbyaddingtothe
wholeNumberoffreePersons,includingthoseboundtoServicefora
TermofYears,andexcludingIndiansnottaxed,threefifthsofallother
Persons.TheactualEnumerationshallbemadewithinthreeYears
afterthefirstMeetingoftheCongressoftheUnitedStates,andwithin
everysubsequentTermoftenYears,insuchMannerastheyshallby
Lawdirect.ThenumberofRepresentativesshallnotexceedonefor
everythirtyThousand,buteachStateshallhaveatLeastone
Rep

In [6]:
data = load_from_wikipedia("GPT-4")

Loading document from WIkipedia for GPT-4


In [7]:
print(data[1].page_content)

GPT-4.5 (codenamed "Orion") is a large language model developed by OpenAI as part of the GPT series. Officially released on February 27, 2025, GPT-4.5 is available to users subscribed to the ChatGPT Plus and Pro plans across web, mobile, and desktop platforms. Access was also provided through the OpenAI API and Developer Playground until July 14, 2025. On August 7, 2025, with the release of GPT-5, GPT-4.5 was removed from both the API and ChatGPT users with the Plus and Teams tier, while Pro users are still able to access to GPT-4.5 model under the "Legacy Models" tab.


== Overview ==
GPT-4.5 was primarily trained using unsupervised learning, which improves its ability to recognize patterns, draw connections, and generate creative insights without reasoning. This method was combined with supervised fine-tuning and reinforcement learning from human feedback. The computational resources needed for training were provided by Microsoft Azure.
Sam Altman described GPT-4.5 as a "giant, expen

### 2. Chunking the loaded documents

In [8]:
data = load_documents("files/us_constitution.pdf")
from langchain_text_splitters import RecursiveCharacterTextSplitter
def chunk_data(data, chunk_size=256):
    print(f"Chunking the data into chunk size {chunk_size}")
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size)
    chunks = splitter.split_documents(data)

    return chunks

chunks = chunk_data(data)

Loading PDF document: files/us_constitution
Chunking the data into chunk size 256


In [9]:
print(len(chunks))

513


In [10]:
# Calculate the cost of embeddings of all the chunks
def price_embedding_cost(texts):
    import tiktoken
    enc = tiktoken.encoding_for_model("text-embedding-ada-002")
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    print(f"Total tokens: {total_tokens}")
    print(f"Cost: ${total_tokens / 1000 * 0.0004:.6f}")
price_embedding_cost(chunks)

Total tokens: 34048
Cost: $0.013619


### Embeddings and uploading to a Vector Database (Pinecone)

In [15]:
def insert_or_fetch_embeddings(index_name, chunks):
    import pinecone # Used to connect to pinecone directly
    from langchain_community.vectorstores import Pinecone # DEPRICATED
    from langchain_pinecone import PineconeVectorStore
    from langchain_openai import OpenAIEmbeddings
    from pinecone import ServerlessSpec

    pc = pinecone.Pinecone()
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1536)
    # If Index already exists in Pinecone just retireve the vector store using langchain -> Pinecone
    if index_name in pc.list_indexes().names():
        print(f"Index {index_name} already exists. Loading embeddings ...", end='')
        vector_store = PineconeVectorStore.from_existing_index(index_name, embeddings)
        print("OK")
        return vector_store
    else:
        print(f"Creatng index {index_name} and embeddings ...", end='')
        pc.create_index(
            name=index_name,
            dimension=1536,
            metric="cosine",
            spec=ServerlessSpec(cloud='aws', region='us-east-1')
        )
        vector_store = PineconeVectorStore.from_documents(chunks, embeddings, index_name=index_name)
        print("OK")
        return vector_store

In [16]:
def delete_pinecone_index(index_name="all"):
    import pinecone
    pc = pinecone.Pinecone()
    if index_name == "all":
        indexes = pc.list_indexes().names()
        print("Deleting all indexes ...")
        for index in indexes:
            pc.delete_index(index)
        print("OK")
    else:
        print(f"Deleting index {index_name} ...", end='')
        pc.delete_index(index_name)
        print("OK")

In [17]:
# Cleaning pinecone before adding more indexes
delete_pinecone_index()

Deleting all indexes ...
OK


In [18]:
index_name = "askadocument"
vector_store = insert_or_fetch_embeddings(index_name, chunks)

Creatng index askadocument and embeddings ...OK
